# 第 7 章：神经网络

本 Notebook 比较线性 Ridge 基准与轻量多层感知机。数据严格按时间切分，所有预处理参数只在训练集上估计。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("../data/data_ml_web.csv.gz")
data = pd.read_csv(DATA_PATH, parse_dates=["date"])

features = [
    "Mkt_Cap_12M_Usd",
    "Pb",
    "Vol1Y_Usd",
    "Mom_11M_Usd",
]
split_date = pd.Timestamp("2016-01-01")

train = data[data["date"] < split_date]
test = data[data["date"] >= split_date]
X_train, y_train = train[features], train["R1M_Usd"]
X_test, y_test = test[features], test["R1M_Usd"]

print(f"训练集：{len(train):,} 条")
print(f"测试集：{len(test):,} 条")

## 线性基准

复杂模型首先需要证明它能在样本外击败一个合理、稳定的简单模型。

In [ ]:
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
ridge_mae = mean_absolute_error(y_test, ridge_pred)
print(f"Ridge 测试集 MAE：{ridge_mae:.5f}")

## 轻量多层感知机

修改隐藏层规模、正则化强度或迭代次数，再重新运行本单元格。

In [ ]:
hidden_layers = (12, 6)
regularization = 0.001
max_iterations = 60

mlp = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=hidden_layers,
        activation="relu",
        alpha=regularization,
        max_iter=max_iterations,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=42,
    ),
)

mlp.fit(X_train, y_train)
mlp_pred = mlp.predict(X_test)
mlp_mae = mean_absolute_error(y_test, mlp_pred)

comparison = pd.Series({
    "Ridge": ridge_mae,
    "MLP": mlp_mae,
}, name="测试集 MAE")
comparison.round(5)

In [ ]:
ax = comparison.sort_values().plot.bar(
    color=["#2878b5", "#f2a541"],
    figsize=(7, 4),
)
ax.set(
    title="样本外误差比较",
    xlabel="模型",
    ylabel="平均绝对误差（越低越好）",
)
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()

## 完整本地版本的下一步

原书使用 TensorFlow/Keras。完整迁移时会在量化服务器建立锁定的 TensorFlow 环境，并补充全量数据、训练检查点、日志和可选 GPU 配置。